## Patient-Adaptive Seizure Prediction using Transfer Learning

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import gc

print("✅ Setup complete")
print("GPU available:", torch.cuda.is_available())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup complete
GPU available: False


In [3]:
!pip install gdown

In [4]:
import gdown

# Your file link
url = "https://drive.google.com/file/d/1x4une6F6S0Bwf8Oe0gPX_Hba4J6c2lhH/view?usp=drive_link"

# Extract file ID and download
file_id = "1x4une6F6S0Bwf8Oe0gPX_Hba4J6c2lhH"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "EEG_Scaled_data.csv", quiet=False)

print("✅ File downloaded as EEG_Scaled_data.csv")

Downloading...
From (original): https://drive.google.com/uc?id=1x4une6F6S0Bwf8Oe0gPX_Hba4J6c2lhH
From (redirected): https://drive.google.com/uc?id=1x4une6F6S0Bwf8Oe0gPX_Hba4J6c2lhH&confirm=t&uuid=0990d280-321a-421e-8d54-10e41aef5080
To: /content/EEG_Scaled_data.csv
100%|██████████| 2.65G/2.65G [00:34<00:00, 75.8MB/s]

✅ File downloaded as EEG_Scaled_data.csv


In [5]:
file_path = "/content/EEG_Scaled_data.csv"

print("Loading dataset...")

# Chunked loading to be safe
chunks = []
for chunk in pd.read_csv(file_path, chunksize=20000):
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

print(f"✅ Dataset loaded! Shape: {df.shape}")
print(f"Target distribution: {df['target'].value_counts().to_dict()}")

Loading dataset...
✅ Dataset loaded! Shape: (11233, 36865)
Target distribution: {0: 9799, 1: 1434}


In [6]:
df.head()

,Channel_1,Channel_2,Channel_3,Channel_4,Channel_5,Channel_6,Channel_7,Channel_8,Channel_9,Channel_10,...,Channel_36856,Channel_36857,Channel_36858,Channel_36859,Channel_36860,Channel_36861,Channel_36862,Channel_36863,Channel_36864,target
0,0.647,0.149,-0.213,-0.199,-0.287,-0.187,0.320,0.445,0.482,0.348,...,0.076,0.283,0.365,-0.013,-0.029,0.223,0.221,0.175,0.347,0
1,-2.450,-2.788,-2.387,-1.370,-1.032,-1.037,-1.253,-1.702,-2.116,-2.019,...,0.154,-0.023,-0.085,0.178,0.118,-0.030,0.048,0.113,0.085,0
2,-0.026,-0.123,-0.347,-0.348,0.027,0.162,0.166,-0.045,-0.083,-0.037,...,0.086,0.262,0.319,0.028,0.144,0.255,0.148,0.082,0.133,0
3,-0.067,-0.153,-0.180,-0.210,-0.238,-0.183,-0.147,-0.238,-0.305,-0.187,...,-0.101,-0.247,-0.271,-0.216,-0.144,-0.066,0.048,-0.127,-0.296,0
4,-0.190,-0.299,-0.333,-0.199,0.182,0.261,0.262,-0.075,-0.403,-0.232,...,0.621,0.625,0.262,0.127,0.301,0.287,0.148,0.191,0.106,0


In [7]:
# Simulate patients
np.random.seed(42)
num_patients = 25
df['patient_id'] = np.random.randint(0, num_patients, size=len(df))

# Use 5000 features
num_features = 5000
feature_cols = [col for col in df.columns if col not in ['target', 'patient_id']][:num_features]

X = df[feature_cols].values.astype(np.float32)
y = df['target'].values.astype(np.int64)
groups = df['patient_id'].values

print(f"Using {num_features} features and {num_patients} pseudo-patients")

Using 5000 features and 25 pseudo-patients


In [8]:
class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx].unsqueeze(0), self.y[idx]

print("Dataset class ready")

Dataset class ready


In [9]:
import torch
import torch.nn as nn

# Load your saved baseline model
baseline_path = "/content/drive/MyDrive/baseline_model.pth"

class BaselineBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=5, stride=2)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=5, stride=2)
        self.pool = nn.MaxPool1d(2)
        self.flatten = nn.Flatten()

    def forward(self, x):
        # Removed x = x.unsqueeze(1) as the dataset already provides a channel dimension
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        return self.flatten(x)

# Create backbone and load weights
backbone = BaselineBackbone()
state_dict = torch.load(baseline_path, map_location=torch.device('cpu'), weights_only=True)
backbone.load_state_dict(state_dict, strict=False)

print("✅ Baseline backbone loaded successfully")

# Adaptive Model
class AdaptiveModel(nn.Module):
    def __init__(self, backbone, num_patients):
        super().__init__()
        self.backbone = backbone
        for param in self.backbone.parameters():
            param.requires_grad = False   # Freeze backbone

        self.personal_heads = nn.ModuleDict({
            str(p): nn.Sequential(
                nn.Linear(19936, 64), # Corrected input features from 32 * (5000 // 8) to 19936
                nn.ReLU(),
                nn.Linear(64, 2)
            ) for p in range(num_patients)
        })

    def forward(self, x, patient_id):
        features = self.backbone(x)
        head = self.personal_heads[str(patient_id.item())]
        return head(features)

model = AdaptiveModel(backbone, num_patients).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
print("✅ Adaptive model with personalization heads created")

✅ Baseline backbone loaded successfully
✅ Adaptive model with personalization heads created


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gkf = GroupKFold(n_splits=5)

print("Starting patient-adaptive training...")

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    print(f"\n--- Fold {fold+1}/5 ---")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    train_dataset = EEGDataset(X_train, y_train)
    val_dataset = EEGDataset(X_val, y_val)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    # Train only personalization heads
    optimizer = optim.Adam(model.personal_heads.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(5):
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            patient_id = torch.tensor([0]).to(device)  # simplified
            optimizer.zero_grad()
            outputs = model(batch_x, patient_id)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    print(f"Fold {fold+1} completed")

print("\n✅ Patient-Adaptive Training Finished!")

Starting patient-adaptive training...

--- Fold 1/5 ---
Fold 1 completed

--- Fold 2/5 ---
Fold 2 completed

--- Fold 3/5 ---
Fold 3 completed

--- Fold 4/5 ---
Fold 4 completed

--- Fold 5/5 ---
Fold 5 completed

✅ Patient-Adaptive Training Finished!


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11233 entries, 0 to 11232
Columns: 36866 entries, Channel_1 to patient_id
dtypes: float64(36864), int64(2)
memory usage: 3.1 GB


In [17]:
class BetterAdaptiveModel(nn.Module):
    def __init__(self, backbone, num_patients, feature_size=5000):
        super().__init__()
        self.backbone = backbone

        # Unfreeze last layers of backbone
        for param in list(self.backbone.parameters())[-6:]:
            param.requires_grad = True

        # Calculate correct input size for adapter
        # After conv + pool: exactly 19936 for 5000 input features
        adapter_input_size = 19936 # Corrected from 32 * (feature_size // 8)

        self.adapters = nn.ModuleDict({
            str(p): nn.Sequential(
                nn.Linear(adapter_input_size, 128),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, 2)
            ) for p in range(num_patients)
        })

    def forward(self, x, patient_id):
        features = self.backbone(x)
        adapter = self.adapters[str(patient_id.item())]
        return adapter(features)

In [18]:
print("Training Better Patient-Adaptive Model...")

# Re-create model with correct dimensions
model = BetterAdaptiveModel(backbone, num_patients, feature_size=5000).to(device)

# Strong class weighting for seizure class
criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 8.0]).to(device))
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

for epoch in range(8):
    model.train()
    total_loss = 0

    # Train on batches with random patient grouping
    for i in range(0, len(X), 512):   # batch of 512 samples
        end = min(i + 512, len(X))
        batch_x = torch.tensor(X[i:end]).unsqueeze(1).to(device)
        batch_y = torch.tensor(y[i:end]).to(device)
        patient_id = torch.tensor([groups[i]]).to(device)   # use first patient's ID for simplicity

        optimizer.zero_grad()
        outputs = model(batch_x, patient_id)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 2 == 0:
        print(f"Epoch {epoch} completed | Avg Loss: {total_loss/len(X):.4f}")

print("✅ Better personalization training completed")

Training Better Patient-Adaptive Model...
Epoch 0 completed | Avg Loss: 0.0013
Epoch 2 completed | Avg Loss: 0.0008
Epoch 4 completed | Avg Loss: 0.0006
Epoch 6 completed | Avg Loss: 0.0005
✅ Better personalization training completed


In [21]:
# === Corrected BetterAdaptiveModel ===
class BetterAdaptiveModel(nn.Module):
    def __init__(self, backbone, num_patients, feature_size=5000):
        super().__init__()
        self.backbone = backbone

        # Unfreeze last layers
        for param in list(self.backbone.parameters())[-6:]:
            param.requires_grad = True

        # Correct adapter input size (calculated from conv layers)
        adapter_input_size = 19936   # Corrected from 32 * (feature_size // 8) to 19936

        self.adapters = nn.ModuleDict({
            str(p): nn.Sequential(
                nn.Linear(adapter_input_size, 128),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, 2)
            ) for p in range(num_patients)
        })

    def forward(self, x, patient_id):
        features = self.backbone(x)
        adapter = self.adapters[str(patient_id.item())]
        return adapter(features)

# === Re-create model ===
model = BetterAdaptiveModel(backbone, num_patients, feature_size=5000).to(device)
print("✅ Corrected Adaptive Model created")

# === Stronger Training with Class Weighting ===
criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 6.0]).to(device))
optimizer = optim.Adam([
    {'params': model.adapters.parameters(), 'lr': 0.002},
    {'params': model.backbone.parameters(), 'lr': 0.0002}   # very small lr for backbone
], weight_decay=1e-5)

print("Starting improved training...")

for epoch in range(10):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for i in range(0, len(X), 256):        # smaller batch for stability
        end = min(i + 256, len(X))
        batch_x = torch.tensor(X[i:end]).unsqueeze(1).to(device)
        batch_y = torch.tensor(y[i:end]).to(device)
        patient_id = torch.tensor([groups[i]]).to(device)

        optimizer.zero_grad()
        outputs = model(batch_x, patient_id)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / num_batches
    if epoch % 2 == 0:
        print(f"Epoch {epoch:2d} | Avg Loss: {avg_loss:.5f}")

print("✅ Improved personalization training completed")

✅ Corrected Adaptive Model created
Starting improved training...
Epoch  0 | Avg Loss: 0.83563
Epoch  2 | Avg Loss: 0.34191
Epoch  4 | Avg Loss: 0.24826
Epoch  6 | Avg Loss: 0.16995
Epoch  8 | Avg Loss: 0.11619
✅ Improved personalization training completed


In [23]:
print("=== Final Evaluation - Generalized vs Patient-Adaptive ===\n")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Generalized Baseline (using fixed adapter '0')
print("Evaluating Generalized Baseline...")
generalized_preds = []
generalized_labels = []

with torch.no_grad():
    for i in range(0, len(X), 512):
        end = min(i + 512, len(X))
        batch_x = torch.tensor(X[i:end]).unsqueeze(1).to(device)
        batch_y = y[i:end]

        # Use adapter '0' for generalized evaluation
        outputs = model.adapters['0'](model.backbone(batch_x))
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        generalized_preds.extend(preds)
        generalized_labels.extend(batch_y)

gen_acc = accuracy_score(generalized_labels, generalized_preds)
gen_auc = roc_auc_score(generalized_labels, generalized_preds)

print(f"Generalized Model → Accuracy: {gen_acc:.4f} | AUC: {gen_auc:.4f}")

# 2. Patient-Adaptive Model
print("\nEvaluating Patient-Adaptive Model...")
adaptive_preds = []
adaptive_labels = []

with torch.no_grad():
    for i in range(0, len(X), 512):
        end = min(i + 512, len(X))
        batch_x = torch.tensor(X[i:end]).unsqueeze(1).to(device)
        batch_y = y[i:end]
        patient_id = torch.tensor([groups[i]]).to(device)

        outputs = model.adapters[str(patient_id.item())](model.backbone(batch_x))
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        adaptive_preds.extend(preds)
        adaptive_labels.extend(batch_y)

ada_acc = accuracy_score(adaptive_labels, adaptive_preds)
ada_auc = roc_auc_score(adaptive_labels, adaptive_preds)

print(f"Patient-Adaptive Model → Accuracy: {ada_acc:.4f} | AUC: {ada_auc:.4f}")

# Final Comparison
print("\n" + "="*70)
print("FINAL COMPARISON")
print("="*70)
print(f"Generalized Model     : Accuracy = {gen_acc:.4f} | AUC = {gen_auc:.4f}")
print(f"Patient-Adaptive Model: Accuracy = {ada_acc:.4f} | AUC = {ada_auc:.4f}")
print(f"Improvement in AUC    : {ada_auc - gen_auc:.4f}")

if ada_auc > gen_auc + 0.03:
    print("✅ Strong improvement from personalization!")
elif ada_auc > gen_auc:
    print("✅ Slight improvement seen.")
else:
    print("⚠️ No significant improvement yet.")

print("\nNote: Seizure recall is the most important metric in real applications.")

=== Final Evaluation - Generalized vs Patient-Adaptive ===

Evaluating Generalized Baseline...
Generalized Model → Accuracy: 0.9343 | AUC: 0.7969

Evaluating Patient-Adaptive Model...
Patient-Adaptive Model → Accuracy: 0.9287 | AUC: 0.8612

FINAL COMPARISON
Generalized Model     : Accuracy = 0.9343 | AUC = 0.7969
Patient-Adaptive Model: Accuracy = 0.9287 | AUC = 0.8612
Improvement in AUC    : 0.0644
✅ Strong improvement from personalization!

Note: Seizure recall is the most important metric in real applications.


In [25]:
print("=== Final Evaluation with Seizure Recall Focus ===\n")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Generalized Model
print("Evaluating Generalized Baseline...")
gen_preds = []
gen_labels = []

with torch.no_grad():
    for i in range(0, len(X), 512):
        end = min(i + 512, len(X))
        batch_x = torch.tensor(X[i:end]).unsqueeze(1).to(device)
        batch_y = y[i:end]

        outputs = model.adapters['0'](model.backbone(batch_x))
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        gen_preds.extend(preds)
        gen_labels.extend(batch_y)

gen_acc = accuracy_score(gen_labels, gen_preds)
gen_auc = roc_auc_score(gen_labels, gen_preds)
gen_recall_seizure = classification_report(gen_labels, gen_preds, output_dict=True)['1']['recall']

print(f"Generalized Model → Accuracy: {gen_acc:.4f} | AUC: {gen_auc:.4f} | Seizure Recall: {gen_recall_seizure:.4f}")

# Patient-Adaptive Model
print("\nEvaluating Patient-Adaptive Model...")
ada_preds = []
ada_labels = []

with torch.no_grad():
    for i in range(0, len(X), 512):
        end = min(i + 512, len(X))
        batch_x = torch.tensor(X[i:end]).unsqueeze(1).to(device)
        batch_y = y[i:end]
        patient_id = torch.tensor([groups[i]]).to(device)

        outputs = model.adapters[str(patient_id.item())](model.backbone(batch_x))
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        ada_preds.extend(preds)
        ada_labels.extend(batch_y)

ada_acc = accuracy_score(ada_labels, ada_preds)
ada_auc = roc_auc_score(ada_labels, ada_preds)
ada_recall_seizure = classification_report(ada_labels, ada_preds, output_dict=True)['1']['recall']

print(f"Patient-Adaptive Model → Accuracy: {ada_acc:.4f} | AUC: {ada_auc:.4f} | Seizure Recall: {ada_recall_seizure:.4f}")

# Final Comparison
print("\n" + "="*75)
print("FINAL COMPARISON (Focus on Seizure Recall)")
print("="*75)
print(f"Generalized Model     : Acc = {gen_acc:.4f} | AUC = {gen_auc:.4f} | Seizure Recall = {gen_recall_seizure:.4f}")
print(f"Patient-Adaptive Model: Acc = {ada_acc:.4f} | AUC = {ada_auc:.4f} | Seizure Recall = {ada_recall_seizure:.4f}")

print(f"\nImprovement in Seizure Recall : {ada_recall_seizure - gen_recall_seizure:.4f}")

if ada_recall_seizure > gen_recall_seizure:
    print("✅ Good! Personalization improved seizure detection recall.")
else:
    print("⚠️ Seizure recall did not improve yet.")

# ========================== SAVE THE MODEL ==========================
adaptive_model_path = "/content/drive/MyDrive/seizure_prediction_data/adaptive_model_final.pth"
torch.save(model.state_dict(), adaptive_model_path)
print(f"\n✅ Adaptive model saved to Google Drive:")
print(f"   {adaptive_model_path}")

# Also save locally and download
torch.save(model.state_dict(), "/content/adaptive_model_final.pth")
from google.colab import files
files.download("/content/adaptive_model_final.pth")

print("\n🎉 Both models saved successfully!")

=== Final Evaluation with Seizure Recall Focus ===

Evaluating Generalized Baseline...
Generalized Model → Accuracy: 0.9341 | AUC: 0.7947 | Seizure Recall: 0.6074

Evaluating Patient-Adaptive Model...
Patient-Adaptive Model → Accuracy: 0.9274 | AUC: 0.8605 | Seizure Recall: 0.7706

FINAL COMPARISON (Focus on Seizure Recall)
Generalized Model     : Acc = 0.9341 | AUC = 0.7947 | Seizure Recall = 0.6074
Patient-Adaptive Model: Acc = 0.9274 | AUC = 0.8605 | Seizure Recall = 0.7706

Improvement in Seizure Recall : 0.1632
✅ Good! Personalization improved seizure detection recall.

✅ Adaptive model saved to Google Drive:
   /content/drive/MyDrive/seizure_prediction_data/adaptive_model_final.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Both models saved successfully!


In [26]:
# 2. Save Generalized Baseline Model (for comparison)
# Load the original baseline again and save it cleanly
baseline_path_drive = "/content/drive/MyDrive/seizure_prediction_data/baseline_model_final.pth"

# If you still have the original baseline model loaded somewhere, or reload it
# For safety, we'll save a copy of the current backbone as generalized reference
torch.save(model.backbone.state_dict(), baseline_path_drive)
print(f"✅ Generalized baseline backbone saved to Drive: {baseline_path_drive}")

torch.save(model.backbone.state_dict(), "/content/baseline_model_final.pth")
files.download("/content/baseline_model_final.pth")

print("\n🎉 Both models saved successfully in the same folder!")
print("You now have:")
print("   - adaptive_model_final.pth")
print("   - baseline_model_final.pth")

✅ Generalized baseline backbone saved to Drive: /content/drive/MyDrive/seizure_prediction_data/baseline_model_final.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Both models saved successfully in the same folder!
You now have:
   - adaptive_model_final.pth
   - baseline_model_final.pth
